# RAG Retrieval Experiment — Letter-Level (Single Letter)

Evaluate dense retrieval at the **letter level** (e.g. `A`, `K`, `J`),
where the corpus is deduplicated to 25 unique single-letter categories.

Dataset: Belgian SOSA 1920–1930 cause-of-death records.

In [2]:
# Cell 1: Imports + Config
from __future__ import annotations

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import re
import sys
import time
from collections import defaultdict
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Make project root importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from codllm.config import DataSourceConfig
from codllm.data_handler import BELGIUM_MAPPING, load_source_dataset
from codllm import icd10h_registry

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Paths ---
MASTERLIST_PATH = PROJECT_ROOT / "data" / "ICD10h_Masterlist_2024.xlsx"

# --- Parameters ---
K_VALUES = [1, 3, 5, 10, 15, 25]
DENSE_MODEL = "intfloat/multilingual-e5-large"
SEED = 42
np.random.seed(SEED)

print(f"Device:       {DEVICE}" + (f" ({torch.cuda.get_device_name()})" if DEVICE == "cuda" else ""))
print(f"Project root: {PROJECT_ROOT}")
print(f"K values:     {K_VALUES}")
print(f"Model:        {DENSE_MODEL}")

ImportError: numpy._core.multiarray failed to import

In [ ]:
# Cell 2: Load Belgian data via existing data handler
belgium_source = DataSourceConfig(
    source_id="belgium_1920_1930",
    path="SOSA_EXTR_1920-1930 (belgium).xlsx",
    mapping_id="belgium",
)

df_belgium = load_source_dataset(
    source=belgium_source,
    mapping=BELGIUM_MAPPING,
    training_input=["cod"],
    max_labels=6,
    data_raw_dir=str(PROJECT_ROOT / "data" / "raw"),
)

# Strip the "cod: " prefix to get raw query text
df_belgium["query"] = df_belgium["text"].str.replace(r"^cod:\s*", "", regex=True)
# Gold code sets — convert to letter level (first character only)
df_belgium["gold_set"] = df_belgium["y_codes"].apply(
    lambda codes: {c[0] for c in codes}
)

# Subsample 10% for faster iteration
SAMPLE_FRAC = 0.10
df_belgium = df_belgium.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)

queries = df_belgium["query"].tolist()
gold_sets = df_belgium["gold_set"].tolist()
record_ids = df_belgium["record_id"].tolist()

print(f"Belgian dataset (10% sample): {len(df_belgium)} records")
print(f"Gold letter codes per record: min={df_belgium['gold_set'].str.len().min()}, "
      f"max={df_belgium['gold_set'].str.len().max()}, "
      f"mean={df_belgium['gold_set'].str.len().mean():.2f}")
df_belgium[["record_id", "query", "gold_set"]].head()

Belgian dataset (10% sample): 4212 records
Gold letter codes per record: min=1, max=4, mean=1.13


,record_id,query,gold_set
0,920A0220,Péritonite généralisée,{K}
1,923A1622,Aangeboren zwakte,{P}
2,925A1200,Pneumonie hypostatique,{J}
3,924A3871,Hémorrhagie cérébrale,{I}
4,923A0059,Convulsions,{R}


In [ ]:
# Cell 3: Load masterlist + build LETTER-LEVEL corpus
masterlist_df = pd.read_excel(MASTERLIST_PATH, sheet_name="Masterlist", engine="openpyxl")
print(f"Masterlist rows: {len(masterlist_df)}")

# Populate the registry
valid_codes = icd10h_registry.load_masterlist(MASTERLIST_PATH)
icd10h_registry.set_valid_codes(valid_codes)

# Build letter corpus: group by first character, combine descriptions
letter_groups: dict[str, dict] = defaultdict(
    lambda: {"descs": set(), "cats": set(), "causes": set(), "histcats": set()}
)

for _, row in masterlist_df.iterrows():
    code = str(row["ICD10h"]).strip()
    if not re.match(r"^[A-Z]\d{2}\.\d{3}$", code):
        continue
    letter = code[0]
    cat = str(row.get("icd10_2levelCATEGORY", "")).strip()
    cause = str(row.get("ICD10_2levelCAUSE", "")).strip()
    histcat = str(row.get("HistCat", "")).strip()

    # For letter-level, use category names rather than individual descriptions
    # (individual descriptions would be thousands of entries concatenated)
    if cat and cat != "nan":
        letter_groups[letter]["cats"].add(cat)
    if cause and cause != "nan":
        letter_groups[letter]["causes"].add(cause)
    if histcat and histcat != "nan":
        letter_groups[letter]["histcats"].add(histcat)

corpus_codes: list[str] = []
corpus_texts: list[str] = []

for letter in sorted(letter_groups):
    g = letter_groups[letter]
    parts = [f"{letter}: {'; '.join(sorted(g['cats']))}"]
    if g["causes"]:
        parts.append(f"Cause: {'; '.join(sorted(g['causes']))}")
    if g["histcats"]:
        parts.append(f"HistCat: {'; '.join(sorted(g['histcats']))}")
    corpus_codes.append(letter)
    corpus_texts.append(" | ".join(parts))

code_to_idx = {c: i for i, c in enumerate(corpus_codes)}
idx_to_code = {i: c for i, c in enumerate(corpus_codes)}
print(f"Letter corpus size: {len(corpus_texts)} letters")
print(f"Letters: {corpus_codes}")
print(f"\nExample corpus entries (truncated):")
for i in range(min(5, len(corpus_texts))):
    print(f"  {corpus_texts[i][:150]}...")

# Sanity check: verify gold codes exist in corpus
all_gold = set()
for gs in gold_sets:
    all_gold.update(gs)
missing = all_gold - set(corpus_codes)
print(f"\nUnique gold letters: {len(all_gold)}")
print(f"Gold letters missing from corpus: {len(missing)}")
if missing:
    print(f"  Missing: {sorted(missing)}")

Masterlist rows: 14088
Letter corpus size: 25 letters
Letters: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y', 'Z']

Example corpus entries (truncated):
  A: Actinomycosis; Acute poliomyelitis; Amoebiasis; Anogenital herpesviral [herpes simplex] infections; Anthrax; Arenaviral haemorrhagic fever; Atypica...
  B: Acute hepatitis A; Acute hepatitis B; African trypanosomiasis; Ascariasis; Aspergillosis; Blastomycosis; Candidiasis; Chagas' disease; Chromomycosi...
  C: Follicular [nodular] non-Hodgkin's lymphoma; Hodgkin's disease; Kaposi's sarcoma; Leukaemia of unspecified cell type; Lymphoid leukaemia; Malignant...
  D: Acquired haemolytic anaemia; Acquired pure red cell aplasia [erythroblastopenia]; Acute posthaemorrhagic anaemia; Agranulocytosis; Anaemia due to e...
  E: Adrenogenital disorders; Amyloidosis; Ascorbic acid deficiency; Congenital iodine-deficiency syndrome; Cushing's syndrome; Cystic fibrosis; Defic

In [ ]:
# Cell 4: Dense Retriever (E5-large only)


class DenseRetriever:
    """Dense retrieval using sentence-transformers + FAISS."""

    def __init__(
        self,
        model_name: str,
        batch_size: int = 64,
        device: str | None = None,
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.is_e5 = "e5" in model_name.lower()
        self.model = SentenceTransformer(model_name, device=device)
        self.index: faiss.IndexFlatIP | None = None
        self.corpus_embeddings: np.ndarray | None = None

    def index_corpus(self, texts: list[str]) -> float:
        """Encode corpus and build FAISS index. Returns encoding time in seconds."""
        corpus_input = [f"passage: {t}" for t in texts] if self.is_e5 else texts
        start = time.time()
        self.corpus_embeddings = self.model.encode(
            corpus_input,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        encode_time = time.time() - start

        # Build FAISS inner-product index
        dim = self.corpus_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.corpus_embeddings.astype(np.float32))
        return encode_time

    def query_batch(
        self, texts: list[str], k: int
    ) -> list[list[tuple[int, float]]]:
        """Retrieve top-k for a batch of queries."""
        query_input = [f"query: {t}" for t in texts] if self.is_e5 else texts
        q_embs = self.model.encode(
            query_input,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype(np.float32)
        scores, indices = self.index.search(q_embs, k)
        results = []
        for i in range(len(texts)):
            results.append([(int(indices[i, j]), float(scores[i, j])) for j in range(k)])
        return results


print("DenseRetriever defined.")

DenseRetriever defined.


In [ ]:
# Cell 5: Index E5-large corpus (letter-level)
print(f"Indexing E5-large on letter corpus ({len(corpus_texts)} entries)...")
e5_retriever = DenseRetriever(DENSE_MODEL, device=DEVICE)
encode_time = e5_retriever.index_corpus(corpus_texts)
print(f"Corpus encode time: {encode_time:.1f}s")

Indexing E5-large on letter corpus (25 entries)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 781.50it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]

Corpus encode time: 1.6s


In [ ]:
# Cell 6: Evaluation functions


def coverage_at_k(gold: set[str], candidates: list[str]) -> float:
    """Fraction of gold codes found in the candidate list."""
    if not gold:
        return 1.0
    return len(gold & set(candidates)) / len(gold)


def all_in_at_k(gold: set[str], candidates: list[str]) -> bool:
    """True if all gold codes are in the candidate list."""
    return gold.issubset(set(candidates))


def f1_per_record(gold: set[str], candidates: list[str]) -> dict:
    """Compute precision, recall, F1 for a single record.

    Treats as multi-label: each code is 'retrieved' or not, 'gold' or not.
    TP = gold codes found in candidates
    FP = candidates not in gold
    FN = gold codes not in candidates
    """
    cand_set = set(candidates)
    tp = len(gold & cand_set)
    fp = len(cand_set - gold)
    fn = len(gold - cand_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


def evaluate_retriever(
    retriever,
    queries: list[str],
    gold_sets: list[set[str]],
    k_values: list[int],
    idx_to_code: dict[int, str],
) -> tuple[pd.DataFrame, list[dict]]:
    """Evaluate a retriever across multiple K values.

    Returns (summary_df, per_record_results).
    """
    max_k = max(k_values)
    start = time.time()
    results = retriever.query_batch(queries, max_k)
    query_time = time.time() - start

    per_record: list[dict] = []
    summary_rows: list[dict] = []

    for k in k_values:
        coverages = []
        all_ins = []
        # Accumulators for micro-F1
        total_tp, total_fp, total_fn = 0, 0, 0
        record_f1s = []

        for i, (gold, result) in enumerate(zip(gold_sets, results)):
            candidate_codes = [idx_to_code[idx] for idx, _ in result[:k]]
            cov = coverage_at_k(gold, candidate_codes)
            ai = all_in_at_k(gold, candidate_codes)
            f1_info = f1_per_record(gold, candidate_codes)

            coverages.append(cov)
            all_ins.append(ai)
            record_f1s.append(f1_info["f1"])
            total_tp += f1_info["tp"]
            total_fp += f1_info["fp"]
            total_fn += f1_info["fn"]

            per_record.append({
                "record_idx": i,
                "k": k,
                "coverage": cov,
                "all_in": ai,
                "n_gold": len(gold),
                "precision": f1_info["precision"],
                "recall": f1_info["recall"],
                "f1": f1_info["f1"],
            })

        # Micro F1: aggregate TP/FP/FN then compute
        micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
        micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
        micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

        # Macro F1: average per-record F1
        macro_f1 = np.mean(record_f1s)

        coverages_arr = np.array(coverages)
        all_ins_arr = np.array(all_ins)
        summary_rows.append({
            "K": k,
            "Mean_Cov": coverages_arr.mean(),
            "Median_Cov": np.median(coverages_arr),
            "Pct_Zero": (coverages_arr == 0).mean() * 100,
            "Pct_Partial": ((coverages_arr > 0) & (coverages_arr < 1)).mean() * 100,
            "Pct_Full": (coverages_arr == 1.0).mean() * 100,
            "AllIn_Pct": all_ins_arr.mean() * 100,
            "Micro_F1": micro_f1,
            "Macro_F1": macro_f1,
        })

    return pd.DataFrame(summary_rows), per_record, query_time

print("Evaluation functions defined.")

Evaluation functions defined.


In [ ]:
# Cell 7: Run E5-large evaluation (letter-level)
print("Evaluating E5-large across K values (letter codes)...")
start = time.time()
summary_df, per_record_list, query_time = evaluate_retriever(
    e5_retriever, queries, gold_sets, K_VALUES, idx_to_code
)
print(f"Query time: {query_time:.1f}s")

summary_df["Model"] = "E5-large"
per_record_df = pd.DataFrame(per_record_list)
per_record_df["model"] = "E5-large"

# --- Results ---
print("\n" + "=" * 70)
print("E5-large Dense Retrieval Results (Letter-Level)")
print("=" * 70)
display(summary_df[["K", "Mean_Cov", "Median_Cov", "Pct_Zero", "Pct_Full", "AllIn_Pct", "Micro_F1", "Macro_F1"]].round(4))

# --- Per-gold-size breakdown ---
print("\n" + "=" * 70)
print("Coverage@K Breakdown by Gold-Set Size")
print("=" * 70)

per_record_df["gold_group"] = per_record_df["n_gold"].apply(
    lambda n: f"{n}" if n < 5 else "5+"
)

group_counts = per_record_df[per_record_df["k"] == K_VALUES[0]].groupby("gold_group").size()
print(f"\nRecords per gold-size group:")
print(group_counts.to_string())

pivot = per_record_df.pivot_table(
    index="gold_group",
    columns="k",
    values="coverage",
    aggfunc="mean",
)
print(f"\nMean coverage by gold-set size and K:")
display(pivot.round(4))

Evaluating E5-large across K values (letter codes)...


Batches: 100%|██████████| 66/66 [00:06<00:00, 10.60it/s]


Query time: 6.3s

E5-large Dense Retrieval Results (Letter-Level)


,K,Mean_Cov,Median_Cov,Pct_Zero,Pct_Full,AllIn_Pct,Micro_F1,Macro_F1
0,1,0.3953,0.0,56.6714,35.8262,35.8262,0.4074,0.4078
1,3,0.6414,1.0,32.1700,60.4463,60.4463,0.3447,0.3415
2,5,0.7553,1.0,21.4150,72.4596,72.4596,0.2747,0.2722
3,10,0.9303,1.0,5.4606,91.4767,91.4767,0.1871,0.1858
4,15,0.9676,1.0,2.7540,96.2488,96.2488,0.1351,0.1343
5,25,1.0000,1.0,0.0000,100.0000,100.0000,0.0863,0.0859



Coverage@K Breakdown by Gold-Set Size

Records per gold-size group:
gold_group
1    3694
2     502
3      15
4       1

Mean coverage by gold-set size and K:


k,1,3,5,10,15,25
gold_group,,,,,,
1,0.4085,0.6538,0.7656,0.9388,0.9686,1.0
2,0.3028,0.5558,0.6853,0.8695,0.9622,1.0
3,0.2444,0.4889,0.5778,0.8667,0.9111,1.0
4,0.2500,0.2500,0.7500,1.0000,1.0000,1.0


In [ ]:
# Cell 7b: Inclusion Rate — E5-large vs Popularity Baseline
import matplotlib.pyplot as plt
from collections import Counter

# --- Popularity baseline ---
# Count how often each letter appears as a gold label across all records
letter_counts = Counter()
for gold in gold_sets:
    letter_counts.update(gold)

# Rank letters by popularity (most frequent first)
letters_by_popularity = [letter for letter, _ in letter_counts.most_common()]

print("Letter popularity ranking:")
for rank, letter in enumerate(letters_by_popularity, 1):
    print(f"  {rank:>2d}. {letter}  ({letter_counts[letter]} records)")

# --- Compute AllIn@K for both strategies ---
N_LETTERS = len(corpus_codes)
K_GRAPH = [1, 2, 3, 5, 7, 10, 15, 20, N_LETTERS]
max_k_graph = max(K_GRAPH)

# E5-large retrieval
graph_results = e5_retriever.query_batch(queries, max_k_graph)

allin_retriever = []
allin_popularity = []

for k in K_GRAPH:
    # Retriever
    ret_count = 0
    for gold, result in zip(gold_sets, graph_results):
        candidate_codes = {idx_to_code[idx] for idx, _ in result[:k]}
        if gold.issubset(candidate_codes):
            ret_count += 1
    allin_retriever.append(ret_count / len(gold_sets) * 100)

    # Popularity baseline: top-K most common letters
    top_k_popular = set(letters_by_popularity[:k])
    pop_count = sum(1 for gold in gold_sets if gold.issubset(top_k_popular))
    allin_popularity.append(pop_count / len(gold_sets) * 100)

# --- Plot ---
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(K_GRAPH, allin_retriever, marker="o", linewidth=2, markersize=6,
        color="#2563eb", label="E5-large (dense retrieval)")
ax.plot(K_GRAPH, allin_popularity, marker="s", linewidth=2, markersize=6,
        color="#dc2626", linestyle="--", label="Top-K most popular letters")
ax.set_xlabel("K (number of letter categories)", fontsize=12)
ax.set_ylabel("Inclusion Rate — AllIn@K (%)", fontsize=12)
ax.set_title(f"E5-large vs Popularity Baseline  (letter-level, {N_LETTERS} letters)", fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xticks(K_GRAPH)
ax.legend(fontsize=11)

# Annotate both lines
for x, yr, yp in zip(K_GRAPH, allin_retriever, allin_popularity):
    ax.annotate(f"{yr:.1f}%", (x, yr), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8, color="#2563eb")
    ax.annotate(f"{yp:.1f}%", (x, yp), textcoords="offset points",
                xytext=(0, -14), ha="center", fontsize=8, color="#dc2626")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Example inspection
rng = np.random.RandomState(SEED)
sample_indices = rng.choice(len(queries), size=min(5, len(queries)), replace=False)
sample_queries = [queries[i] for i in sample_indices]
sample_results = e5_retriever.query_batch(sample_queries, N_LETTERS)

for j, idx in enumerate(sample_indices):
    print("=" * 80)
    print(f"Record {record_ids[idx]}")
    print(f"Query: {queries[idx]}")
    print(f"Gold letters: {sorted(gold_sets[idx])}")
    print(f"\nFull ranking ({N_LETTERS} letters, E5-large):")
    for rank, (cidx, score) in enumerate(sample_results[j], 1):
        code = idx_to_code[cidx]
        in_gold = "*" if code in gold_sets[idx] else " "
        print(f"  {rank:>2d}. [{in_gold}] {code}  (score={score:.4f})")
    print()

Batches: 100%|██████████| 1/1 [00:00<00:00, 25.40it/s]

Record 928A0805
Query: Etter pleuris, streptokokken
Gold letters: ['A', 'J']

Full ranking (25 letters, E5-large):
   1. [*] J  (score=0.8280)
   2. [*] A  (score=0.8250)
   3. [ ] K  (score=0.8187)
   4. [ ] I  (score=0.8168)
   5. [ ] G  (score=0.8109)
   6. [ ] E  (score=0.8092)
   7. [ ] R  (score=0.8071)
   8. [ ] W  (score=0.8064)
   9. [ ] B  (score=0.8052)
  10. [ ] L  (score=0.8051)
  11. [ ] N  (score=0.8041)
  12. [ ] H  (score=0.8025)
  13. [ ] P  (score=0.8023)
  14. [ ] M  (score=0.8018)
  15. [ ] S  (score=0.7992)
  16. [ ] O  (score=0.7981)
  17. [ ] T  (score=0.7980)
  18. [ ] Y  (score=0.7967)
  19. [ ] Q  (score=0.7942)
  20. [ ] Z  (score=0.7931)
  21. [ ] X  (score=0.7924)
  22. [ ] D  (score=0.7910)
  23. [ ] C  (score=0.7875)
  24. [ ] F  (score=0.7775)
  25. [ ] V  (score=0.7622)

Record 922A3842
Query: Long tering
Gold letters: ['A']

Full ranking (25 letters, E5-large):
   1. [*] A  (score=0.7706)
   2. [ ] O  (score=0.7685)
   3. [ ] G  (score=0.7683)
   4. [

In [ ]:
# Cell 9: Speed stats
print("Speed Statistics (Letter-Level)")
print(f"  Corpus size: {len(corpus_texts)} letter entries")
print(f"  Corpus encode: {encode_time:.1f}s")
print(f"  Query time ({len(queries)} queries): {query_time:.1f}s")
print(f"  Avg per query: {query_time / len(queries) * 1000:.1f}ms")

Speed Statistics (Letter-Level)
  Corpus size: 25 letter entries
  Corpus encode: 1.6s
  Query time (4212 queries): 6.3s
  Avg per query: 1.5ms


In [ ]:
# Cell 10: Save artifacts
OUT_DIR = PROJECT_ROOT / "data" / "cache" / "rag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

comparison_csv = OUT_DIR / "letter_comparison_results.csv"
summary_df.to_csv(comparison_csv, index=False)
print(f"Saved comparison table: {comparison_csv}")

per_record_pkl = OUT_DIR / "letter_per_record_results.pkl"
per_record_df.to_pickle(per_record_pkl)
print(f"Saved per-record results: {per_record_pkl}")

Saved comparison table: c:\Users\edlun\Desktop\DTU\Bachelor\codLLM\data\cache\rag\letter_comparison_results.csv
Saved per-record results: c:\Users\edlun\Desktop\DTU\Bachelor\codLLM\data\cache\rag\letter_per_record_results.pkl
